In [19]:
import helpers.data as data
import helpers.features as features
import joblib
import hmmlearn.hmm as hmm
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score, classification_report

In [20]:
hmm_features = ["mean_spread", "spread_std", "vol_accel", "har_rv", "mean_compression", "composite_mean"]
# Use only hmm_state for live predictions (no data leakage)
# Previous version used ["hmm_state", "target"] which caused data leakage
gbt_features = ["hmm_state", "mean_divergence", "divergence_change", "mean_velocity", "dist_vah", "dist_val", "composite_mean", "har_rv", "har_sigma", "vol_accel"]
train_size = 0.6
sequence_length = 12


In [21]:
df = data.get_data()
df = features.add_features(df)
df = df.dropna()

Features computed in 1.523s


In [ ]:
hmm = hmm.GaussianHMM(n_components=2, covariance_type="diag")
hmm.fit(df[hmm_features].iloc[:int(len(df) * train_size)])
joblib.dump(hmm, "./trained_models/regime_hmm.pkl")

['./trained_models/regime_hmm.pkl']

In [5]:
hmm = joblib.load("./trained_models/regime_hmm.pkl")
df["hmm_state"] = hmm.predict(df[hmm_features])

In [6]:
df = features.add_target(df)
train_df = df[gbt_features].iloc[:int(len(df) * train_size)]
targets = df['target'].iloc[:int(len(df) * train_size)].values

def create_sequences(data, target_values):
    feature_data = data[gbt_features].values
    X_sequences = []
    y_targets = []
    
    for i in range(sequence_length, len(data)):
        sequence = feature_data[i - sequence_length:i]
        flattened_sequence = sequence.flatten()
        X_sequences.append(flattened_sequence)
        y_targets.append(target_values[i])
    
    return np.array(X_sequences), np.array(y_targets)

X, Y = create_sequences(train_df, targets)
print(f"Training data shape: X={X.shape}, Y={Y.shape}")
print(f"Target stats: mean={Y.mean():.6f}, std={Y.std():.6f}, min={Y.min():.6f}, max={Y.max():.6f}")
print(f"Class distribution: {np.bincount(Y.astype(int))} (0: {np.sum(Y==0)}, 1: {np.sum(Y==1)})")

model = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    subsample=0.7,
    max_features='sqrt',
    loss='log_loss',  # Binary cross-entropy for classification
    validation_fraction=0.2,
    n_iter_no_change=25,
    random_state=40,
    verbose=1
)

model.fit(X, Y)

# Get probability predictions (0-1) for class 1
train_pred_proba = model.predict_proba(X)[:, 1]  # Probability of class 1
train_pred_class = model.predict(X)  # Binary predictions (0 or 1)

# Create test set from remaining data
test_df_full = df.iloc[int(len(df) * train_size):].copy()
test_df_full = test_df_full.dropna(subset=gbt_features + ['target'])
test_df = test_df_full[gbt_features]
test_targets = test_df_full['target'].values

X_test, Y_test = create_sequences(test_df, test_targets)

# Get probability predictions for test set
test_pred_proba = model.predict_proba(X_test)[:, 1]  # Probability of class 1
test_pred_class = model.predict(X_test)  # Binary predictions (0 or 1)

# Classification metrics
train_acc = accuracy_score(Y, train_pred_class)
test_acc = accuracy_score(Y_test, test_pred_class)
train_log_loss = log_loss(Y, train_pred_proba)
test_log_loss = log_loss(Y_test, test_pred_proba)
train_auc = roc_auc_score(Y, train_pred_proba)
test_auc = roc_auc_score(Y_test, test_pred_proba)

print(f"\nTraining Metrics:")
print(f"  Accuracy: {train_acc:.6f}")
print(f"  Log Loss: {train_log_loss:.6f}")
print(f"  ROC AUC: {train_auc:.6f}")

print(f"\nTest Metrics:")
print(f"  Accuracy: {test_acc:.6f}")
print(f"  Log Loss: {test_log_loss:.6f}")
print(f"  ROC AUC: {test_auc:.6f}")

print(f"\nPrediction probabilities - Train: [{train_pred_proba.min():.6f}, {train_pred_proba.max():.6f}]")
print(f"Prediction probabilities - Test: [{test_pred_proba.min():.6f}, {test_pred_proba.max():.6f}]")
print(f"Target range - Train: [{Y.min():.6f}, {Y.max():.6f}]")
print(f"Target range - Test: [{Y_test.min():.6f}, {Y_test.max():.6f}]")

print(f"\nClassification Report (Test):")
print(classification_report(Y_test, test_pred_class, target_names=['Class 0', 'Class 1']))

# Feature importance analysis
feature_importance = model.feature_importances_
# Since we have sequences, group by original features
n_features_per_timestep = len(gbt_features)
feature_names_expanded = [f"{feat}_t-{i}" for i in range(sequence_length-1, -1, -1) for feat in gbt_features]
top_indices = np.argsort(feature_importance)[-20:][::-1]
print(f"\nTop 20 Most Important Features:")
for idx in top_indices:
    print(f"  {feature_names_expanded[idx]}: {feature_importance[idx]:.6f}")

# Aggregate importance by original feature
feature_importance_agg = np.zeros(len(gbt_features))
for i, feat in enumerate(gbt_features):
    start_idx = i * sequence_length
    end_idx = (i + 1) * sequence_length
    feature_importance_agg[i] = feature_importance[start_idx:end_idx].sum()

print(f"\nAggregated Feature Importance:")
sorted_feat_idx = np.argsort(feature_importance_agg)[::-1]
for idx in sorted_feat_idx:
    print(f"  {gbt_features[idx]}: {feature_importance_agg[idx]:.6f}")

joblib.dump(model, "./trained_models/gbt.pkl")
print(f"\nModel saved. Best iteration: {model.n_estimators_}")

Training data shape: X=(41346, 120), Y=(41346,)
Target stats: mean=0.072897, std=0.259967, min=0.000000, max=1.000000
Class distribution: [38332  3014] (0: 38332, 1: 3014)
      Iter       Train Loss      OOB Improve   Remaining Time 
         1           0.5141           0.0079           22.83s
         2           0.4941          -0.0145           23.39s
         3           0.4923           0.0281           23.44s
         4           0.4869           0.0076           23.36s
         5           0.4682          -0.0211           23.18s
         6           0.4651           0.0114           23.12s
         7           0.4719           0.0315           22.85s
         8           0.4605          -0.0111           22.94s
         9           0.4625           0.0190           23.19s
        10           0.4571           0.0013           23.17s
        20           0.4377           0.0314           22.27s
        30           0.4126          -0.0042           21.48s
        40           